In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
df = pd.read_csv("../data/courses.csv")
df

,year,faculties,title,code,credits,evaluation_methods,second_chance
0,2025-26,"['IC Education, trajectschijf 3', 'IC Nursing ...",Interprofessional collaboration,189483,3 studiepunten,"[[{'moment': 'Binnen examenrooster', 'format':...",wel mogelijk.
1,2025-26,"['IC AgroFood Sustainability', 'IC Audiology',...",Survival Dutch,189484,3 studiepunten,"[[{'moment': 'Buiten examenrooster', 'format':...",wel mogelijk.
2,2025-26,"['IC Education, trajectschijf 3']",Freinet methodology,189485,6 studiepunten,"[[{'moment': 'Binnen en buiten examenrooster',...",wel mogelijk.
3,2025-26,"['IC Education, trajectschijf 3']",Study visits: innovational education,189486,3 studiepunten,"[[{'moment': 'Binnen examenrooster', 'format':...",wel mogelijk.
4,2025-26,"['IC Education, trajectschijf 3']",Socially engaged teacher,189487,3 studiepunten,"[[{'moment': 'Binnen examenrooster', 'format':...",wel mogelijk.
...,...,...,...,...,...,...,...
11335,2025-26,"['Mobiliteitsopleiding DLO/DSA/DGZ, trajectsch...",Adapted physical activity,202369,5 studiepunten,NaN,NaN
11336,2025-26,"['Mobiliteitsopleiding DLO/DSA/DGZ, trajectsch...",Future factory project,202370,10 studiepunten,NaN,NaN
11337,2025-26,"['Master in de muziek, trajectschijf 1']",Kunst in het werkveld (deel 1) (IB),202373,4 studiepunten,NaN,NaN
11338,2025-26,"['Master in de muziek, trajectschijf 1']",Kunst in het werkveld (deel 2) (IB),202374,4 studiepunten,NaN,NaN


In [3]:
df["faculties"]

0        ['IC Education, trajectschijf 3', 'IC Nursing ...
1        ['IC AgroFood Sustainability', 'IC Audiology',...
2                        ['IC Education, trajectschijf 3']
3                        ['IC Education, trajectschijf 3']
4                        ['IC Education, trajectschijf 3']
                               ...                        
11335    ['Mobiliteitsopleiding DLO/DSA/DGZ, trajectsch...
11336    ['Mobiliteitsopleiding DLO/DSA/DGZ, trajectsch...
11337             ['Master in de muziek, trajectschijf 1']
11338             ['Master in de muziek, trajectschijf 1']
11339    ['Mobiliteitsopleiding School of Arts Internsh...
Name: faculties, Length: 11340, dtype: object

In [171]:
# Unique values in faculties
# set(faculty for sublist in df["faculties"].dropna().apply(eval) for faculty in sublist)
#!/usr/bin/env python3

In [4]:
# Make new row for each faculty in faculties
# Faculty looks like "['IC AgroFood Sustainability', 'IC Audiology', 'IC Business, Retail and Languages', 'IC Chemistry, Biotechnology and Environmental Technology', 'IC Education, trajectschijf 3', 'IC Fashion & Textile Technology, trajectschijf 1', 'IC IT', 'IC Nursing (niet in 2025-26)', 'IC Nutrition and Dietetics (niet in 2025-26)', 'IC Occupational Therapy', 'IC Speech and Language Pathology', 'IC Sports program', 'IC Wood Technology']"
import ast

df["faculties"] = df["faculties"].apply(ast.literal_eval)
df_exploded = df.explode("faculties", ignore_index=True)
df = df_exploded.copy()

# Split faculty and program_stage, program_stage comes after a comma
df[["faculty", "program_stage"]] = df["faculties"].str.split(", trajectschijf ", expand=True)

# Get rid of "studiepunten" in credits column and convert to float
df["credits"] = df["credits"].str.replace("studiepunten", "")
df["credits"] = df["credits"].str.replace("studiepunt", "").astype(int)

# Map unique values in "second_chance" to true/false
df["second_chance"] = df["second_chance"].map({"wel mogelijk.": True, "niet mogelijk.": False}).astype(bool)

In [5]:
# Show all columns that contain null values
df.isnull().sum()[df.isnull().sum() > 0]

evaluation_methods    8588
program_stage         2167
dtype: int64

In [6]:
df

,year,faculties,title,code,credits,evaluation_methods,second_chance,faculty,program_stage
0,2025-26,"IC Education, trajectschijf 3",Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Education,3
1,2025-26,IC Nursing (niet in 2025-26),Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Nursing (niet in 2025-26),None
2,2025-26,IC Nutrition and Dietetics (niet in 2025-26),Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Nutrition and Dietetics (niet in 2025-26),None
3,2025-26,IC Occupational Therapy,Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Occupational Therapy,None
4,2025-26,IC AgroFood Sustainability,Survival Dutch,189484,3,"[[{'moment': 'Buiten examenrooster', 'format':...",True,IC AgroFood Sustainability,None
...,...,...,...,...,...,...,...,...,...
12066,2025-26,"Mobiliteitsopleiding DLO/DSA/DGZ, trajectschijf 3",Adapted physical activity,202369,5,NaN,True,Mobiliteitsopleiding DLO/DSA/DGZ,3
12067,2025-26,"Mobiliteitsopleiding DLO/DSA/DGZ, trajectschijf 3",Future factory project,202370,10,NaN,True,Mobiliteitsopleiding DLO/DSA/DGZ,3
12068,2025-26,"Master in de muziek, trajectschijf 1",Kunst in het werkveld (deel 1) (IB),202373,4,NaN,True,Master in de muziek,1
12069,2025-26,"Master in de muziek, trajectschijf 1",Kunst in het werkveld (deel 2) (IB),202374,4,NaN,True,Master in de muziek,1


In [32]:
# Evaluation method where moment is "Binnen examenrooster"
df["evaluation_methods"]

0      [{'moment': 'Binnen examenrooster', 'format': ...
1      [{'moment': 'Binnen examenrooster', 'format': ...
2      [{'moment': 'Binnen examenrooster', 'format': ...
3      [{'moment': 'Binnen examenrooster', 'format': ...
4      [{'moment': 'Binnen examenrooster', 'format': ...
                             ...                        
157    [{'moment': 'Binnen examenrooster', 'format': ...
158    [{'moment': 'Binnen examenrooster', 'format': ...
159    [{'moment': 'Buiten examenrooster', 'format': ...
160    [{'moment': 'Buiten examenrooster', 'format': ...
161    [{'moment': 'Buiten examenrooster', 'format': ...
Name: evaluation_methods, Length: 162, dtype: object

In [14]:
df

,year,faculties,title,code,credits,evaluation_methods,second_chance,faculty,program_stage
0,2025-26,"IC Education, trajectschijf 3",Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Education,3
1,2025-26,IC Nursing (niet in 2025-26),Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Nursing (niet in 2025-26),None
2,2025-26,IC Nutrition and Dietetics (niet in 2025-26),Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Nutrition and Dietetics (niet in 2025-26),None
3,2025-26,IC Occupational Therapy,Interprofessional collaboration,189483,3,"[[{'moment': 'Binnen examenrooster', 'format':...",True,IC Occupational Therapy,None
4,2025-26,IC AgroFood Sustainability,Survival Dutch,189484,3,"[[{'moment': 'Buiten examenrooster', 'format':...",True,IC AgroFood Sustainability,None
...,...,...,...,...,...,...,...,...,...
12066,2025-26,"Mobiliteitsopleiding DLO/DSA/DGZ, trajectschijf 3",Adapted physical activity,202369,5,NaN,True,Mobiliteitsopleiding DLO/DSA/DGZ,3
12067,2025-26,"Mobiliteitsopleiding DLO/DSA/DGZ, trajectschijf 3",Future factory project,202370,10,NaN,True,Mobiliteitsopleiding DLO/DSA/DGZ,3
12068,2025-26,"Master in de muziek, trajectschijf 1",Kunst in het werkveld (deel 1) (IB),202373,4,NaN,True,Master in de muziek,1
12069,2025-26,"Master in de muziek, trajectschijf 1",Kunst in het werkveld (deel 2) (IB),202374,4,NaN,True,Master in de muziek,1


In [ ]:
# Write to DimClass table
import sqlalchemy as sa
server = "127.0.0.1,1500"
database = "DEP2"
username = "sa"
password = "dep2025-G12"
driver = "ODBC Driver 17 for SQL Server"

engine = sa.create_engine(
    f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}"
)

with engine.connect() as conn:
    df_dim_class = df.copy()[["title", "code", "credits", "second_chance", "program_stage"]]
    df_dim_class = df_dim_class.rename(
        columns={
            "title": "ClassName",
            "code": "ClassCode",
            "credits": "ClassCredits",
            "second_chance": "ClassSecondChance",
            "program_stage": "ClassProgramStage"
        }
    )
    df_dim_class["ClassKey"] = range(1, len(df_dim_class) + 1)
    df_dim_class.to_sql("DimClass", conn, if_exists="append", index=False)